# 1. Import library

In [8]:
import os
import random
import datetime
import pandas   as pd
import numpy    as np

from tqdm                       import tqdm
from preProcessingData          import PreProcessData
from utility.matpilotDraw       import MatPlot
from prettytable                import PrettyTable
import umap
import plotly.express           as px

matPlot    = MatPlot() 
preData    = PreProcessData()

# 2. Preprocessing

In [9]:
preData.dataLoading()

Process:42944||2024-04-17 17:21:25,962-INFO [preProcessingData.py:dataLoading():174]:Loading data from files ...
Process:42944||2024-04-17 17:23:08,756-INFO [preProcessingData.py:dataLoading():180]:Finishing loading data ...


Index(['tiempo', 'Descripción Equipo', 'Modo de Falla', 'Comentario',
       'Especialidad', 'Motivo', 'Inicio Reportado', 'Fin Reportado',
       'corriente(A)', 'velocidad(rpm)', 'nivel_tolva_compensacion(%)',
       'Potencia(kW)', 'posicion_poste(%)', 'rendimiento(ton)',
       'presion_estanque_aire(bar)', 'Piñón_ext(°C)', 'Piñón_int(°C)',
       'Piñón_int(mm/s)', 'Piñón_ext(mm/s)', 'Eje_piñón_aceite(%)',
       'Lubricación_retorno(°C)', 'Lubricación_depósito(°C)',
       'Lubricación_entrada(°C)', 'Lubricación_principal(psi)',
       'Lubricación_principal(A)', 'Lubricación_respaldo(psi)',
       'Lubricación_respaldo(A)', 'Lubricación_post_filtros(psi)',
       'Lubricación_válvula_MPS(psi)', 'Lubricación_entrada_sup(psi)',
       'Lubricación_entrada_inf(psi)', 'Lubricación_entrada_sup(m3/h)',
       'Lubricación_entrada_inf(m3/h)', 'Lubricación_depósito(%)',
       'Motor_Fase1(A)', 'Motor_Fase2(A)', 'Motor_Fase3(A)', 'Motor_Fase1(V)',
       'Motor_Fase2(V)', 'Motor_Fase3(V

#### Remove redundant mode of failure (option)

In [ ]:
preData.rmRedundantMOF()

#### Manipulated and efficiency Variables

In [13]:
preData.splitInputandEfficiencyVar()

+-------------------------------+-----------------------------+
|         ManupulatedVar        |        EfficiencyVar        |
+-------------------------------+-----------------------------+
|          corriente(A)         | nivel_tolva_compensacion(%) |
|         velocidad(rpm)        |       rendimiento(ton)      |
|          Potencia(kW)         |        Piñón_ext(°C)        |
|       posicion_poste(%)       |        Piñón_int(°C)        |
|   presion_estanque_aire(bar)  |       Piñón_int(mm/s)       |
|      Eje_piñón_aceite(%)      |       Piñón_ext(mm/s)       |
|   Lubricación_principal(psi)  |   Lubricación_retorno(°C)   |
|    Lubricación_principal(A)   |   Lubricación_depósito(°C)  |
|   Lubricación_respaldo(psi)   |   Lubricación_entrada(°C)   |
|    Lubricación_respaldo(A)    |                             |
| Lubricación_post_filtros(psi) |                             |
|  Lubricación_válvula_MPS(psi) |                             |
|  Lubricación_entrada_sup(psi) |       

##### Draw variables

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

#### Sumarry MOF

In [14]:
preData.sumarryMOF()

+-------------------------------------------------------------------------------------------------------+-----------------------------------+-----------------------------------+
|                                              Elimination                                              |            Critical MOF           |              Used MOF             |
+-------------------------------------------------------------------------------------------------------+-----------------------------------+-----------------------------------+
|                                                Stand by                                               |         Válvula defectuosa        |         Válvula defectuosa        |
|                                        Stock de producto lleno                                        |       Mantención programada       |      Lubricación  deficiente      |
|                                            Cambio de Turno                                            |     

#### Data summarization

In [16]:
preData.sumarryDataFromFile()

All the mode of failure:
Válvula defectuosa
Mantención programada
Falta de aceite hidráulico
Lubricación  deficiente
Temperatura fuera de rango
Bomba hidráulica con fuga
Fuga de aceite
Falla en sistema hidráulico
Falla sistema de hidroset-poste
Mandibulas o revestimiento suelto
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| No |                MOF                |     startingTime    |      endingTime     | staringRow | endingRow |
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| 1  |         Válvula defectuosa        | 2023-01-06 10:52:00 | 2023-01-06 15:13:00 |    4776    |    5037   |
| 2  |       Mantención programada       | 2023-01-19 00:54:00 | 2023-01-19 01:53:00 |   16978    |   17037   |
| 3  |       Mantención programada       | 2023-01-19 03:24:00 | 2023-01-19 23:12:00 |   17128    |   18316   |
| 4  |       Mantención programada       | 2023-0

## TRAIN VAE for reconstruction error

### D. UMAP for entire dataset

In [ ]:
manipulatedCr1  = preData.crusher1[preData.manipulatedCr1]
maWTimeCr1      = manipulatedCr1[:]
print("The input variables shape",maWTimeCr1.shape)
umapReduction = umap.UMAP(  n_components=3,
                            n_neighbors=15,
                            min_dist=0.1)
maEmbedding       = umapReduction.fit_transform(maWTimeCr1)

In [ ]:
efficiencyCr1   = preData.crusher1[preData.efficiencyCr1]
effWTimeCr1     = efficiencyCr1[:]
print("The efficieny variables shape",effWTimeCr1.shape)

umapReduction1 = umap.UMAP(  n_components=3,
                            n_neighbors=15,
                            min_dist=0.1)

effEmbedding      = umapReduction1.fit_transform(effWTimeCr1)


#### c.1 Draw UMAP

In [ ]:
matPlot.draw3D(yCoor1 = maEmbedding[:,0],
               yCoor2 = maEmbedding[:,1],
               yCoor3 = maEmbedding[:,2],
               title  = "UMAP for Manipulated variables")

matPlot.draw3D(yCoor1 = effEmbedding[:,0],
               yCoor2 = effEmbedding[:,1],
               yCoor3 = effEmbedding[:,2],
               title  = "UMAP for efficient variables")

In [ ]:
maEmbedding_df  = pd.DataFrame(maEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
effEmbedding_df = pd.DataFrame(effEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
print(maEmbedding_df)
print(effEmbedding_df)
fig = px.scatter_3d(
    maEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()

fig = px.scatter_3d(
    effEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()

